### Load data set

In [12]:

import numpy as np
from sklearn.datasets import load_wine

data = load_wine()
X = data.data
y = data.target
n_classes = len(np.unique(y))
print('no of class', n_classes)
data

no of class 3


{'data': array([[1.423e+01, 1.710e+00, 2.430e+00, ..., 1.040e+00, 3.920e+00,
         1.065e+03],
        [1.320e+01, 1.780e+00, 2.140e+00, ..., 1.050e+00, 3.400e+00,
         1.050e+03],
        [1.316e+01, 2.360e+00, 2.670e+00, ..., 1.030e+00, 3.170e+00,
         1.185e+03],
        ...,
        [1.327e+01, 4.280e+00, 2.260e+00, ..., 5.900e-01, 1.560e+00,
         8.350e+02],
        [1.317e+01, 2.590e+00, 2.370e+00, ..., 6.000e-01, 1.620e+00,
         8.400e+02],
        [1.413e+01, 4.100e+00, 2.740e+00, ..., 6.100e-01, 1.600e+00,
         5.600e+02]], shape=(178, 13)),
 'target': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

Train test split and scaling

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Binarize labels for ROC
y_test_bin = label_binarize(y_test, classes=[0,1,2])

## SVM with gridserachcv

In [15]:
from sklearn.metrics import roc_auc_score
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

svm_params = {
    'C':[0.1,1,10],
    'gamma':[0.1,0.01],
    'kernel':['rbf','linear']
}
# here cv means cross validation, its 5 fold cross validation
svm_grid = GridSearchCV(SVC(probability=True), svm_params, cv=5)
svm_grid.fit(X_train,y_train)

svm_best = svm_grid.best_estimator_
svm_probs = svm_best.predict_proba(X_test)

svm_auc = roc_auc_score(y_test_bin, svm_probs, multi_class='ovr')

## Naive Bayes

In [16]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X_train,y_train)

nb_probs = nb.predict_proba(X_test)

nb_auc = roc_auc_score(y_test_bin, nb_probs, multi_class='ovr')


## Logistic with GridSearch

In [17]:
from sklearn.linear_model import LogisticRegression

log_params = {
    'C':[0.01,0.1,1,10],
    'solver':['lbfgs']
}

log_grid = GridSearchCV(LogisticRegression(max_iter=200), log_params, cv=5)
log_grid.fit(X_train,y_train)

log_best = log_grid.best_estimator_
log_probs = log_best.predict_proba(X_test)

log_auc = roc_auc_score(y_test_bin, log_probs, multi_class='ovr')

### Find who is the best for iris dataset

In [18]:
# Print Results
print("Best SVM Params:", svm_grid.best_params_)
print("SVM ROC AUC:", svm_auc)

print("Naive Bayes ROC AUC:", nb_auc)

print("Best Logistic Params:", log_grid.best_params_)
print("Logistic Regression ROC AUC:", log_auc)

Best SVM Params: {'C': 1, 'gamma': 0.1, 'kernel': 'linear'}
SVM ROC AUC: 0.9990379990379991
Naive Bayes ROC AUC: 1.0
Best Logistic Params: {'C': 0.1, 'solver': 'lbfgs'}
Logistic Regression ROC AUC: 1.0
